Обучение нейронных сетей на центральном процессоре занимает много времени, особенно в том случае, если нейронную сеть обучают для решения задач компьютерного зрения. Поэтому для тренировки моделей используют графические процессоры, которые существенно ускоряют процесс. 


Специально для обучения моделей компьютерного зрения мы разработали новый тренажёр. Все вычисления будут производиться на сервере с графической картой (GPU) Yandex Compute Cloud.


В этот курс входят только задачи на обучение моделей на серверах с GPU. Процесс обучения занимает время, поэтому специалисты по Data Science сначала запускают тренировку, а затем переходят к другим задачам, пока модель обучается. 


После того как отправите модель на обучение в тренажёре, мы предлагаем вернуться к основному курсу и продолжить изучение уроков о компьютерном зрении. Чтобы вам не пришлось переходить из курса в курс слишком часто, мы добавили необходимую информацию из основного курса «Компьютерное зрение» в уроки с заданиями.

In [1]:
pip install tensorflow

Note: you may need to restart the kernel to use updated packages.


In [17]:
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Conv2D, Flatten, Dense, AvgPool2D, GlobalAveragePooling2D
import numpy as np
from tensorflow import keras
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.preprocessing.image import ImageDataGenerator 
from tensorflow.keras.applications.resnet import ResNet50

## 1. Обучение многослойной сети [GPU]

Постройте и обучите свёрточную нейронную сеть на наборе данных с одеждой. Для этого создайте в коде три функции:
1. загрузки обучающей выборки load_train(),
2. создания модели create_model(),
3. запуска модели train_model().


Добейтесь того, чтобы значение accuracy на тестовой выборке было не меньше 85%.

In [18]:
def load_train(path):
    features_train = np.load(path + 'train_features.npy')
    target_train = np.load(path + 'train_target.npy')
    features_train = features_train.reshape(features_train.shape[0], 28 * 28) / 255.
    return features_train, target_train


def create_model(input_shape):
    model = keras.models.Sequential()
    model.add(Dense(100, input_shape=input_shape, activation='relu'))
    model.add(Dense(30, activation='relu'))
    model.add(Dense(10, activation='softmax'))
    model.compile(optimizer='sgd', loss='sparse_categorical_crossentropy',
                  metrics=['acc'])
    return model


def train_model(model, train_data, test_data, batch_size=32, epochs=10,
               steps_per_epoch=None, validation_steps=None):

    features_train, target_train = train_data
    features_test, target_test = test_data
    model.fit(features_train, target_train, 
              validation_data=(features_test, target_test),
              batch_size=batch_size, epochs=epochs,
              steps_per_epoch=steps_per_epoch,
              validation_steps=validation_steps,
              verbose=2, shuffle=True)
    return model 

## 2. Алгоритм Adam

Постройте и обучите свёрточную нейронную сеть на наборе данных с одеждой. Для этого создайте в коде три функции:
1. загрузки обучающей выборки load_train(),
2. создания модели create_model(),
3. запуска модели train_model().


Добейтесь того, чтобы значение accuracy на тестовой выборке было не меньше 87%.

In [19]:
def load_train(path):
    features_train = np.load(path + 'train_features.npy')
    target_train = np.load(path + 'train_target.npy')
    features_train = features_train.reshape(-1, 28, 28, 1) / 255.0
    return features_train, target_train


def create_model(input_shape):
    model = keras.models.Sequential()
    model.add(Conv2D(filters=6, kernel_size=(5, 5), padding='same',activation="relu", input_shape=(28, 28, 1)))
    model.add(AvgPool2D(pool_size=(2, 2)))
    model.add(Conv2D(filters=16, kernel_size=(5, 5), strides=2, activation="relu"))
    model.add(AvgPool2D(pool_size=(2, 2)))
    model.add(Flatten())
    model.add(Dense(units=100, activation='relu')) 
    model.add(Dense(units=50, activation='relu')) 
    model.add(Dense(units=10, activation='softmax')) 
    optimizer = Adam(lr=0.001) 
    model.compile(optimizer=optimizer, loss='sparse_categorical_crossentropy', metrics=['acc']) 
    return model


def train_model(model, train_data, test_data, batch_size=32, epochs=20,
               steps_per_epoch=None, validation_steps=None):

    features_train, target_train = train_data
    features_test, target_test = test_data
    model.fit(features_train, target_train, 
              validation_data=(features_test, target_test),
              batch_size=batch_size, epochs=epochs,
              steps_per_epoch=steps_per_epoch,
              validation_steps=validation_steps,
              verbose=2, shuffle=True)
    return model 

## 3. Свёрточные сети для классификации фруктов

Постройте и обучите свёрточную нейронную сеть на наборе данных с фруктами. Для этого создайте в коде три функции:
1. загрузки обучающей выборки load_train() (теперь она вернёт загрузчик данных),
2. создания модели create_model(),
3. запуска модели train_model().


Добейтесь того, чтобы значение **accuracy** на тестовой выборке было не меньше **90%**.


У вас есть ограничение: **модель должна обучиться за час**.


Для выполнения задания будет использована полная версия датасета с фруктами. Путь к ней уже указан в прекоде.

In [20]:
def load_train(path):
    datagen = ImageDataGenerator(validation_split=0.25, rescale=1./255)
    train_datagen_flow = datagen.flow_from_directory(
    path,
    target_size=(150, 150),
    batch_size=16,
    class_mode='sparse',
    subset='training',
    seed=12345)
    return train_datagen_flow


def create_model(input_shape):
    model = keras.models.Sequential()
    model.add(Conv2D(filters=6, kernel_size=(5, 5), padding='same',activation="relu", input_shape=input_shape))
    model.add(AvgPool2D(pool_size=(2, 2)))
    model.add(Conv2D(filters=16, kernel_size=(5, 5), strides=2, activation="relu"))
    model.add(AvgPool2D(pool_size=(2, 2)))
    model.add(Flatten())
    model.add(Dense(units=120, activation='relu')) 
    model.add(Dense(units=84, activation='relu')) 
    model.add(Dense(units=12, activation='softmax')) 
    optimizer = Adam(lr=0.0001) 
    model.compile(optimizer=optimizer, loss='sparse_categorical_crossentropy', metrics=['acc']) 
    return model


def train_model(model, train_data, test_data, batch_size=None, epochs=12,
               steps_per_epoch=None, validation_steps=None):
    
    model.fit(train_data, 
              validation_data=test_data,
              batch_size=batch_size, epochs=epochs,
              steps_per_epoch=steps_per_epoch,
              validation_steps=validation_steps,
              verbose=2, shuffle=True)
    return model 

## 4. ResNet в Keras

Постройте и обучите архитектуру ResNet на наборе данных с фруктами. 


Добейтесь того, чтобы значение accuracy на тестовой выборке было не меньше 99%.

    
У вас есть ограничение: модель должна обучиться за полчаса.

In [21]:
def load_train(path):
    datagen = ImageDataGenerator(validation_split=0.25, rescale=1./255)
    train_datagen_flow = datagen.flow_from_directory(
    path,
    target_size=(150, 150),
    batch_size=16,
    class_mode='sparse',
    subset='training',
    seed=12345)
    return train_datagen_flow


def create_model(input_shape):
    backbone = ResNet50(input_shape=input_shape,
                    weights='/datasets/keras_models/resnet50_weights_tf_dim_ordering_tf_kernels_notop.h5',
                    include_top=False)
    model = Sequential()
    model.add(backbone)
    model.add(GlobalAveragePooling2D())
    model.add(Dense(12, activation='softmax')) 
    optimizer = Adam(lr=0.0001) 
    model.compile(optimizer=optimizer, loss='sparse_categorical_crossentropy', metrics=['acc']) 
    return model


def train_model(model, train_data, test_data, batch_size=None, epochs=5,
               steps_per_epoch=None, validation_steps=None):
    
    model.fit(train_data, 
              validation_data=test_data,
              batch_size=batch_size, epochs=epochs,
              steps_per_epoch=steps_per_epoch,
              validation_steps=validation_steps,
              verbose=2, shuffle=True)
    return model 

## 5. Обучение модели

Обучите модель в GPU-тренажёре и сохраните результат вывода модели на экран.


Но прежде ознакомьтесь с несколькими рекомендациями:


Функцией потерь не обязательно должна быть MAE. Зачастую нейронные сети с функцией потерь MSE обучаются быстрее.


Качество на валидационной выборке улучшается, но модель при этом переобучается всё сильнее? Не спешите менять модель. Обычно нейронные сети с большим числом слоёв сильно переобучаются.


Проверьте, что методы load_train(path) и load_test(path) корректно работают с данными. В папке path содержится csv-файл labels.csv с двумя колонками file_name и real_age и папка с изображениями /final_files. Прочитайте данные из csv-файла labels.csv в датафрейм, который будет одним из параметров метода ImageDataGenerator —flow_from_dataframe(dataframe, directory, ...).


Сначала ваш код должен пройти предварительную проверку, а затем его поставят в очередь на обучение. Когда пройдёт 2–3 часа, загляните в этот урок и проверьте, не завершилось ли обучение модели. Когда модель обучится, можете продолжить работу над проектом. Вас ждёт заключительная часть — анализ модели.

Постройте и обучите свёрточную нейронную сеть на датасете с фотографиями людей. Добейтесь значения MAE на тестовой выборке не больше 8.
Функцию загрузки тестовой выборки load_test(path) напишите самостоятельно. Вместе со старыми функциями в коде должны быть:
1. load_train(path),
2. load_test(path),
3. create_model(input_shape),
4. train_model(model, train_data, test_data, batch_size, epochs, steps_per_epoch, validation_steps).


В статье о датасете, с которым вы работаете, значение MAE равно 5.4 — если вы получите MAE меньше 7, это будет отличный результат!

In [22]:
def load_train(path):
    train_data=pd.read_csv(path + 'labels.csv')
    train_datagen = ImageDataGenerator(
                                       validation_split=0.25,
                                       rescale=1./255
                                       )
    train_datagen_flow = train_datagen.flow_from_dataframe(
                                                        dataframe=train_data,
                                                        directory=path + '/final_files/',
                                                        x_col='file_name',
                                                        y_col='real_age',
                                                        target_size=(224, 224),
                                                        batch_size=32,
                                                        class_mode='raw',
                                                        subset='training',
                                                        seed=12345)
    return train_datagen_flow  

def load_test(path):
    test_data=pd.read_csv(path + 'labels.csv')
    test_datagen = ImageDataGenerator(
                                       validation_split=0.25,
                                       rescale=1./255,
                                       )
    test_datagen_flow = test_datagen.flow_from_dataframe(
                                                        dataframe=test_data,
                                                        directory=path + '/final_files/',
                                                        x_col='file_name',
                                                        y_col='real_age',
                                                        target_size=(224, 224),
                                                        batch_size=32,
                                                        class_mode='raw',
                                                        subset='validation',
                                                        seed=12345)
    return test_datagen_flow

def create_model(input_shape):
    backbone = ResNet50(input_shape=input_shape,
                        weights='imagenet',
                        include_top=False)
    model = Sequential()
    model.add(backbone)
    model.add(GlobalAveragePooling2D())
    model.add(Dense(1, activation='relu'))
    optimizer = Adam(lr=0.0001)
    model.compile(loss='mse', optimizer=optimizer, metrics=['mae'])
    return model

def train_model(model, train_data, test_data, batch_size=None, epochs=10,
               steps_per_epoch=None, validation_steps=None):

    if steps_per_epoch is None:
        steps_per_epoch = len(train_data)
    if validation_steps is None:
        validation_steps = len(test_data)
        
    model.fit(train_data,
              validation_data=test_data,
              batch_size=batch_size, epochs=epochs,
              steps_per_epoch=steps_per_epoch,
              validation_steps=validation_steps,
              verbose=2,
              shuffle=True)
    return model

## Вывод



В проекте была обучена модель ResNet50 для определения возраста человека по фотографии.

Использовалась предобученная архитектура ResNet50 без верхних слоёв.

К модели был добавлен слой GlobalAveragePooling2D и полносвязный выходной слой для регрессии.



В качестве функции потерь использовалась MSE, а качество оценивалось по MAE.

Итоговое значение MAE составило 7 лет, что удовлетворяет условию задачи MAE ≤ 8.



Модель в среднем ошибается примерно на 7 лет.